In [0]:
import requests
from azure.storage.blob import BlobServiceClient, ContentSettings
from datetime import datetime, timezone
import json

timestamp = datetime.now(timezone.utc).strftime(
    "%Y%m%d_%H%M%S_%f"
)

connection_string = dbutils.secrets.get(
    scope="azure_store_key",
    key="connection_string"
)


tfl_key = dbutils.secrets.get(
    scope="api_keys",
    key="tfl_key"
)

blob_service_client = BlobServiceClient.from_connection_string(
    connection_string
)



In [0]:
def blob_client(file_name, blob_service_client):
    return (
        blob_service_client.get_blob_client(
        container="project1try",
        blob=file_name
        )
    )

def get_response(url):
    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        return response
    except requests.RequestException as e:
        print(f"Fehler: {e}")
        return None

In [0]:
url_line = "https://api.tfl.gov.uk/Line/Route"

response = get_response(url = url_line)

file_name = f"api_data/line_data/response_{timestamp}.json"

blob_client_line = blob_client(file_name=file_name, blob_service_client=blob_service_client)

blob_client_line.upload_blob(
    response.content,
    overwrite=False,
    content_settings=ContentSettings(
        content_type="application/json"
    )
)

In [0]:
data = response.json()

tfl_ids = [line["id"] for line in data if line.get("id")]

catch_modes = []

for line in data:
    if line.get("modeName") not in catch_modes:
        catch_modes.append(line.get("modeName"))

tfl_modes = ",".join(catch_modes)

In [0]:

# Retrieve line statuses by transport mode and disruptions by line ID
# to preserve line-level associations in the collected raw datasets.
disruption_data = []
for i in range(len(tfl_ids)):
    url_disruption = f"https://api.tfl.gov.uk/Line/{tfl_ids[i]}/Disruption?app_key={tfl_key}"    
    response = get_response(url=url_disruption)
    if response.status_code == 200:
        data = response.json()
        if data != []:
            for item in data: item["line_id"] = str(tfl_ids[i])
            disruption_data.extend(data)
        
    else:
        print(f"Error: {response.status_code})")
        

json_disruption_data = json.dumps(disruption_data)

file_name = f"api_data/disruption_data/response_{timestamp}.json"

blob_client_disruption = blob_client(file_name=file_name, blob_service_client=blob_service_client)

blob_client_disruption.upload_blob(
    json_disruption_data,
    overwrite=False,
    content_settings=ContentSettings(
        content_type="application/json"
    )
)

In [0]:
url_status = f"https://api.tfl.gov.uk/Line/Mode/{tfl_modes}/Status"

response = get_response(url=url_status)

response.raise_for_status()

file_name = f"api_data/status_data/response_{timestamp}.json"

blob_client_status = blob_client(file_name=file_name, blob_service_client=blob_service_client)

blob_client_status.upload_blob(
    response.content,
    overwrite=False,
    content_settings=ContentSettings(
        content_type="application/json"
    )
)